# DeFoG — Session Initialisation (run at the start of every session)

Restores the full `defog` environment from Google Drive and wires up all paths.
Run all cells top-to-bottom before opening any other notebook.

### Prerequisites
- `01_env_setup.ipynb` has been run at least once (tarball exists on Drive).
- The two precomputed `.pt` files are already in `MyDrive/DeFoGColab/data/qm9/`.
- `WANDB_API_KEY` is saved as a Colab Secret (left sidebar → 🔑 Secrets).

### Expected runtime:  ~6–8 min

In [ ]:
# ── Cell 1: Mount Drive + load W&B API key from Colab Secrets ─────────────────
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

# Reads the key you stored under the name WANDB_API_KEY in Colab Secrets.
# If the secret is missing, training will still work but W&B logging will fail.
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab Secrets.')
except Exception as e:
    print(f'Warning: could not load WANDB_API_KEY ({e}).')
    print('Set it manually: os.environ["WANDB_API_KEY"] = "<your-key>"')

# Suppress matplotlib GUI attempts; graph_tool respects this too.
os.environ['MPLBACKEND'] = 'Agg'

print('Drive mounted.')

In [ ]:
%%bash
# ── Cell 2: Install Miniforge ─────────────────────────────────────────────────
# Miniforge is the conda-forge project's own conda installer.
# It ships pre-configured to use conda-forge only, so no Anaconda
# Terms-of-Service acceptance is ever required (unlike Miniconda ≥ 2024).
# Miniforge itself is not cached — only the defog env tarball is (~30 sec).
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Conda already installed."
    /content/miniconda3/bin/conda --version
    exit 0
fi
wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p /content/miniconda3
rm /tmp/miniforge.sh
echo "Miniforge installed."
/content/miniconda3/bin/conda --version

In [ ]:
%%bash
# ── Cell 3: Restore defog environment from Drive tarball ──────────────────────
# Extracts the conda-pack tarball and fixes hardcoded absolute paths.
# The repo is not cloned yet at this point, so the restore logic is inline
# rather than delegating to colab_restore_env.sh.
# Runtime: ~4-6 min (Drive read + tar extraction + conda-unpack).
set -e

TARBALL='/content/drive/MyDrive/DeFoGColab/defog_env.tar.gz'
ENV_DIR='/content/miniconda3/envs/defog'

if [ ! -f "$TARBALL" ]; then
    echo "ERROR: $TARBALL not found. Run 01_env_setup.ipynb first." >&2
    exit 1
fi

echo "Tarball size: $(du -sh $TARBALL | cut -f1)"
echo "Extracting..."
mkdir -p "$ENV_DIR"
tar -xzf "$TARBALL" -C "$ENV_DIR"
echo "Running conda-unpack..."
"$ENV_DIR/bin/conda-unpack"
echo "Environment restored."
"$ENV_DIR/bin/python" --version

In [ ]:
# ── Cell 4: Configure — set your GitHub repo URL here ─────────────────────────
GITHUB_REPO = 'https://github.com/<YOUR_USERNAME>/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')

In [ ]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
# ── Cell 5: Clone the DeFoG repository ───────────────────────────────────────
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present — pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready."
git -C "$REPO_DIR" log --oneline -3

In [ ]:
%%bash -s "$REPO_DIR"
# ── Cell 6: Wire Drive paths via symlinks ─────────────────────────────────────
#
# checkpoints symlink:  src/checkpoints/ → Drive/DeFoGColab/checkpoints/
#   Every checkpoint write by PyTorch Lightning goes directly to Drive.
#   Checkpoints survive session disconnects automatically.
#
# data symlink:         DeFoGPlus/data/  → Drive/DeFoGColab/data/
#   QM9 raw data downloaded on the first run is persisted to Drive and
#   reused in future sessions without re-downloading.
#   The precomputed .pt files are also visible through this symlink.
REPO_DIR="$1"
set -e

DRIVE_BASE='/content/drive/MyDrive/DeFoGColab'
mkdir -p "$DRIVE_BASE/checkpoints" "$DRIVE_BASE/data/qm9"

# checkpoints symlink (inside src/)
CKPT_LINK="$REPO_DIR/src/checkpoints"
if [ -L "$CKPT_LINK" ]; then
    rm "$CKPT_LINK"
fi
ln -s "$DRIVE_BASE/checkpoints" "$CKPT_LINK"
echo "Symlink: $CKPT_LINK → $DRIVE_BASE/checkpoints"

# data symlink (at repo root level)
DATA_LINK="$REPO_DIR/data"
if [ -L "$DATA_LINK" ] || [ -d "$DATA_LINK" ]; then
    rm -rf "$DATA_LINK"
fi
ln -s "$DRIVE_BASE/data" "$DATA_LINK"
echo "Symlink: $DATA_LINK → $DRIVE_BASE/data"

echo "Symlinks ready."

In [ ]:
# ── Cell 7: Full verification ─────────────────────────────────────────────────
import subprocess, os

PYTHON    = '/content/miniconda3/envs/defog/bin/python'
REPO_DIR  = '/content/DeFoGPlus'
DRIVE_BASE = '/content/drive/MyDrive/DeFoGColab'

# ── 1. Python imports
check = """
import graph_tool, torch, rdkit, torch_geometric, pytorch_lightning, hydra, wandb
print('graph_tool       :', graph_tool.__version__)
print('torch            :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU              :', torch.cuda.get_device_name(0))
print('torch_geometric  :', torch_geometric.__version__)
print('pytorch_lightning:', pytorch_lightning.__version__)
"""
env = {**os.environ, 'MPLBACKEND': 'Agg'}
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=env)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Import check failed.')

# ── 2. W&B auth
r2 = subprocess.run(
    [PYTHON, '-m', 'wandb', 'whoami'],
    capture_output=True, text=True,
    env={**env, 'WANDB_API_KEY': os.environ.get('WANDB_API_KEY', '')}
)
print('W&B identity:', r2.stdout.strip() or '(not logged in)')

# ── 3. Precomputed .pt files
pt_files = [
    f'{REPO_DIR}/data/qm9/qm9_motif_marginals_6x1.pt',
    f'{REPO_DIR}/data/qm9/spminer_motif_marginals.pt',
]
for f in pt_files:
    status = '✓' if os.path.isfile(f) else '✗  MISSING'
    print(f'  {status}  {f}')

# ── 4. Symlinks
for link, target in [
    (f'{REPO_DIR}/src/checkpoints', f'{DRIVE_BASE}/checkpoints'),
    (f'{REPO_DIR}/data',            f'{DRIVE_BASE}/data'),
]:
    ok = os.path.islink(link) and os.readlink(link) == target
    print(f'  {"✓" if ok else "✗"} symlink  {link}')

print('\nSession init complete — ready for 03_run_ablation.ipynb')